In [1]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [2]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [3]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [4]:
# Add Compounds
sim.AddCompound("Water")

In [5]:
energy_stream = sim.AddFlowsheetObject('Energy Stream','energy_stream')
sp_1 = sim.AddFlowsheetObject('Solar Panel','SP-1')

In [6]:
energy_stream = energy_stream.GetAsObject()
sp_1 = sp_1.GetAsObject()

In [7]:
sim.ConnectObjects(sp_1.GraphicObject, energy_stream.GraphicObject,  -1, -1)

Exception: The requested connection between the given objects cannot be done.
   at DWSIM.Drawing.SkiaSharp.GraphicsSurface.ConnectObject(GraphicObject gObjFrom, GraphicObject gObjTo, Int32 fidx, Int32 tidx)
   at DWSIM.FlowsheetBase.FlowsheetBase.ConnectObjects(IGraphicObject gobjfrom, IGraphicObject gobjto, Int32 fromidx, Int32 toidx)
   at System.RuntimeMethodHandle.InvokeMethod(Object target, Void** arguments, Signature sig, Boolean isConstructor)
   at System.Reflection.MethodBaseInvoker.InvokeDirectByRefWithFewArgs(Object obj, Span`1 copyOfArgs, BindingFlags invokeAttr)

In [ ]:
sim.AutoLayout()

In [ ]:
# property package
Thermo_Package_UNIQUAC = sim.CreateAndAddPropertyPackage("Peng-Robinson (PR)")

In [ ]:
# Setting the solar panel parameters
sp_1.PanelArea = 10.0  # m^2
sp_1.PanelEfficiency = 0.15  # 15% efficiency
sp_1.SolarIrradiation_kW_m2 = 1.0  # kW/m^2
sp_1.NumberOfPanels = 1

In [ ]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

In [ ]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/16 Solar Panel Automation/16 Solar Panel Automation.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)